# 1. Data Load

In [243]:
import os
from pathlib import Path

In [244]:
data_path = Path("../data")

movie_path = f"{data_path}/movies.csv"
rating_path = f"{data_path}/ratings.csv"
tag_path = f"{data_path}/tags.csv"
link_path = f"{data_path}/links.csv"

In [245]:
import pandas as pd

movies_df = pd.read_csv(movie_path)
ratings_df = pd.read_csv(rating_path)
tags_df = pd.read_csv(tag_path)
links_df = pd.read_csv(link_path)


## a) Null check

In [246]:
df_list = [movies_df, ratings_df, tags_df]

for df in df_list:
    print(df.isnull().sum(), '\n')

movieId    0
title      0
genres     0
dtype: int64 

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64 

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64 



# b) index check

In [247]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [248]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [249]:
tags_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


movie id, user id 는 속도를 위해 int 형으로 그대로 사용했습니다.

# c) check duplicate

In [250]:
for col in movies_df:
    print(f"{col} : \t", movies_df[col].nunique())

movieId : 	 9742
title : 	 9737
genres : 	 951


In [251]:
duplicated_title = movies_df[movies_df['title'].duplicated()]['title']
movies_df[movies_df['title'].duplicated()]

,movieId,title,genres
5601,26958,Emma (1996),Romance
6932,64997,War of the Worlds (2005),Action|Sci-Fi
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller


In [252]:
dup_temp = movies_df[movies_df['title'].isin(duplicated_title)].sort_values(by='title')
dup_temp

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi


In [253]:
# genres 가 가장 긴 영화 기준으로 중복 데이터 처리
dup_temp['genres_len'] = dup_temp['genres'].str.len()
dup_temp

,movieId,title,genres,genres_len
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,27
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller,35
650,838,Emma (1996),Comedy|Drama|Romance,20
5601,26958,Emma (1996),Romance,7
5854,32600,Eros (2004),Drama,5
9135,147002,Eros (2004),Drama|Romance,13
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller,25
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,15
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller,32
6932,64997,War of the Worlds (2005),Action|Sci-Fi,13


In [254]:
group_temp = dup_temp.groupby(by='title')
dup_title_max_len_idx = group_temp['genres_len'].idxmax() # 데이터 검산을 위한 drop idx 와 max_len_idx → 데이터 삭제 이후에는 인덱스 접근이 아닌 키값으로 접근할 것

movies_df.loc[dup_title_max_len_idx]

,movieId,title,genres
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller


In [255]:
drop_idx = dup_temp.index.difference(dup_title_max_len_idx)
dup_temp.loc[drop_idx]

,movieId,title,genres,genres_len
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,27
5601,26958,Emma (1996),Romance,7
5854,32600,Eros (2004),Drama,5
6932,64997,War of the Worlds (2005),Action|Sci-Fi,13
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,15


In [256]:
drop_movieId = dup_temp.loc[drop_idx]['movieId']
max_len_movieId = dup_temp.loc[dup_title_max_len_idx]['movieId']

In [257]:
movies_refine = movies_df.drop(drop_idx).copy()

In [258]:
print("삭제된 ROW :\t", abs(len(movies_df) - len(movies_refine)))
print("dup index 갯수 :\t", len(drop_idx))

삭제된 ROW :	 5
dup index 갯수 :	 5


In [259]:
movies_refine.loc[dup_title_max_len_idx]

,movieId,title,genres
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller


In [260]:
max_len_movieId

9106    144606
650        838
9135    147002
2141      2851
5931     34048
Name: movieId, dtype: int64

In [261]:
# 중복 제거 확인용
movies_refine[movies_refine['movieId'].isin(max_len_movieId.values)]

,movieId,title,genres
650,838,Emma (1996),Comedy|Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance


In [262]:
movies_refine[movies_refine['movieId'].isin(drop_movieId.values)]

,movieId,title,genres


In [263]:
movies_df = movies_refine

In [264]:
max_len_movieId.values

array([144606,    838, 147002,   2851,  34048])

In [265]:
idx_list = group_temp['movieId'].apply(list)
idx_list

title
Confessions of a Dangerous Mind (2002)     [6003, 144606]
Emma (1996)                                  [838, 26958]
Eros (2004)                               [32600, 147002]
Saturn 3 (1980)                            [2851, 168358]
War of the Worlds (2005)                   [34048, 64997]
Name: movieId, dtype: object

In [266]:
# mapping 을 위한 series 생성
map_series = dup_temp.loc[dup_title_max_len_idx].set_index('title')['movieId']
map_series

title
Confessions of a Dangerous Mind (2002)    144606
Emma (1996)                                  838
Eros (2004)                               147002
Saturn 3 (1980)                             2851
War of the Worlds (2005)                   34048
Name: movieId, dtype: int64

In [267]:
mapping_dict = {}
for title, movie_ids in idx_list.items():
    dict_value = map_series[title]
    for id in movie_ids:
        mapping_dict[id] = dict_value

In [268]:
mapping_dict

{6003: 144606,
 144606: 144606,
 838: 838,
 26958: 838,
 32600: 147002,
 147002: 147002,
 2851: 2851,
 168358: 2851,
 34048: 34048,
 64997: 34048}

In [269]:
# 중복치 제거 확인용
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp
4747,28,64997,3.5,1234850075
11451,68,64997,2.5,1230497715
17449,111,6003,4.0,1516468531
23053,156,6003,3.5,1106882187
26958,182,6003,3.0,1054780821
42984,288,6003,4.0,1066059244
54020,356,6003,4.5,1229139513
59953,387,6003,3.5,1208707060
64063,414,6003,3.5,1092414917
74530,474,6003,3.5,1087831997


In [270]:
ratings_refine = ratings_df['movieId'].map(mapping_dict)

In [271]:
ratings_refine = ratings_refine.fillna(ratings_df['movieId'])

In [272]:
# 정제 데이터를 기존 데이터로 변경
ratings_df['movieId'] = ratings_refine

In [273]:
# 중복 제거 확인2
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp


In [274]:
tags_df[tags_df['movieId'].isin(drop_movieId)]

,userId,movieId,tag,timestamp
2058,474,6003,television,1138307058


In [275]:
tags_refine = tags_df['movieId'].map(mapping_dict)
tags_refine.fillna(tags_df['movieId'])
tags_df['movieId'] = tags_refine
tags_df[tags_df['movieId'].isin(drop_movieId)]

,userId,movieId,tag,timestamp


In [276]:
links_df[links_df['movieId'].isin(drop_movieId)]

,movieId,imdbId,tmdbId
4169,6003,290538,4912.0
5601,26958,118308,12254.0
5854,32600,377059,NaN
6932,64997,449040,34812.0
9468,168358,79285,19761.0


In [277]:
links_refine = links_df['movieId'].map(mapping_dict)
links_refine.fillna(links_df['movieId'])
links_df['movieId'] = tags_refine
links_df[links_df['movieId'].isin(drop_movieId)]

,movieId,imdbId,tmdbId


In [278]:
print('movie_df 의 중복치 제거 확인 (0 이면 제거 완료) :\t', movies_df['movieId'].isin(drop_movieId).sum())
print('rating_df 의 중복치 제거 확인 :\t',ratings_df['movieId'].isin(drop_movieId).sum())
print('tags_df 의 중복치 제거 확인 :\t', tags_df['movieId'].isin(drop_movieId).sum())
print('links_df 의 중복치 제거 확인 :\t', links_df['movieId'].isin(drop_movieId).sum())

movie_df 의 중복치 제거 확인 (0 이면 제거 완료) :	 0
rating_df 의 중복치 제거 확인 :	 0
tags_df 의 중복치 제거 확인 :	 0
links_df 의 중복치 제거 확인 :	 0


# 장르 결측치 처리

https://www.themoviedb.org/

## a) check missing value

In [279]:
print(movies_df['genres'].str.contains('no genres listed').sum(),"개의 장르 결측치 존재")

34 개의 장르 결측치 존재


In [280]:
genres_null = movies_df[movies_df['genres'].str.contains('no genres listed')]
genres_null

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)
8684,122888,Ben-hur (2016),(no genres listed)
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),(no genres listed)
8782,129250,Superfast! (2015),(no genres listed)
8836,132084,Let It Be Me (1995),(no genres listed)
8902,134861,Trevor Noah: African American (2013),(no genres listed)
9033,141131,Guardians (2016),(no genres listed)
9053,141866,Green Room (2015),(no genres listed)
9070,142456,The Brand New Testament (2015),(no genres listed)
9091,143410,Hyena Road,(no genres listed)


In [281]:
# 장르 조사용
# genres_null['title'].to_csv('movie_name_null')

In [282]:
pd.set_option('display.max_colwidth', None)
genres_null

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)
8684,122888,Ben-hur (2016),(no genres listed)
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),(no genres listed)
8782,129250,Superfast! (2015),(no genres listed)
8836,132084,Let It Be Me (1995),(no genres listed)
8902,134861,Trevor Noah: African American (2013),(no genres listed)
9033,141131,Guardians (2016),(no genres listed)
9053,141866,Green Room (2015),(no genres listed)
9070,142456,The Brand New Testament (2015),(no genres listed)
9091,143410,Hyena Road,(no genres listed)


# b) check genres value for replacing Null

In [283]:
exist_genre_index = movies_df.index.difference(genres_null.index)
all_genres = movies_df.loc[exist_genre_index]['genres'].replace(r'\|', ' ', regex=True).str.split(' ').explode().unique()

In [284]:
all_genres

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

# c) fill genre manually

In [285]:
movies_df[movies_df['title'] == 'La cravate (1957)']

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)


In [286]:
movies_df.loc[movies_df['title']=='La cravate (1957)', 'genres']='Fantasy Comedy'


In [287]:
movies_df[movies_df['title'] == 'La cravate (1957)']

,movieId,title,genres
8517,114335,La cravate (1957),Fantasy Comedy


In [288]:
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Pirates of the Caribbean: Dead Men Tell No Tales (2017)', 'genres'] = 'Action Adventure Fantasy'
movies_df.loc[movies_df['title']=='Superfast! (2015)', 'genres'] = 'Comedy Action'
movies_df.loc[movies_df['title']=='Let It Be Me (1995)', 'genres'] = 'Romance'
movies_df.loc[movies_df['title']=='Trevor Noah: African American (2013)', 'genres'] = 'Comedy'
movies_df.loc[movies_df['title']=='Guardians (2016)', 'genres'] = 'Action Adventure Drama' # 확인 필요

# 김태연님
movies_df.loc[movies_df['title']=='The Adventures of Sherlock Holmes and Doctor Watson: The Hunt for the Tiger (1980)', 'genres'] = 'Mystery Crime Adventure'
movies_df.loc[movies_df['title']=='The Putin Interviews (2017)', 'genres'] = 'Documentary War'
movies_df.loc[movies_df['title']=='Black Mirror', 'genres'] = 'Sci-Fi Fantasy Drama Mystery'
movies_df.loc[movies_df['title']=='Too Funny to Fail: The Life and Death of The Dana Carvey Show (2017)', 'genres'] = 'Documentary'
movies_df.loc[movies_df['title']=='Serving in Silence: The Margarethe Cammermeyer Story (1995)', 'genres'] = 'Drama'
movies_df.loc[movies_df['title']=='A Christmas Story Live! (2017)', 'genres'] = 'Musical'
# 홍승혜님
movies_df.loc[movies_df['title']=='Cosmos', 'genres'] = 'Unknown'
movies_df.loc[movies_df['title']=='Maria Bamford: Old Baby', 'genres'] = 'Comedy Documentary'
movies_df.loc[movies_df['title']=='Death Note: Desu nôto (2006-2007)', 'genres'] = 'Animation Mystery Sci-Fi Fantasy'
movies_df.loc[movies_df['title']=='Generation Iron 2', 'genres'] = 'Documentary'
movies_df.loc[movies_df['title']=='T2 3-D: Battle Across Time (1996)', 'genres'] = 'Action Sci-Fi'
movies_df.loc[movies_df['title']=='The Godfather Trilogy: 1972-1990 (1992)	', 'genres'] = 'Crime Drama'

movies_df.loc[movies_df['title']=='A Cosmic Christmas (1977)', 'genres'] = 'Animation Sci-Fi'
movies_df.loc[movies_df['title']=='Green Room (2015)', 'genres'] = 'Action Horror'
movies_df.loc[movies_df['title']=='The Brand New Testament (2015)', 'genres'] = 'Comedy Fantasy'
movies_df.loc[movies_df['title']=='Hyena Road', 'genres'] = 'War Drama Thriller Action'
movies_df.loc[movies_df['title']=='The Adventures of Sherlock Holmes and Doctor Watson', 'genres'] = 'Mystery Crime Adventure'

movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action Adventure Drama'


# d) genre check

In [289]:
temp_df = movies_df.copy() # 코드 재실행시를 위한 temp df 생성

In [290]:
pd.set_option('display.max_colwidth',None)
movies_df['genres'] = temp_df['genres'].replace(r'\|', ' ', regex=True).str.split()
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9739,193585,Flint (2017),[Drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


# e) year data check

In [291]:
print(len(movies_df[~movies_df['title'].str.contains(r'\(')]), '개의 연도 결측치 존재')
temp_for_year = movies_df[~movies_df['title'].str.contains(r'\(')]
temp_for_year

12 개의 연도 결측치 존재


,movieId,title,genres
6059,40697,Babylon 5,[Sci-Fi]
9031,140956,Ready Player One,"[Action, Sci-Fi, Thriller]"
9091,143410,Hyena Road,"[War, Drama, Thriller, Action]"
9138,147250,The Adventures of Sherlock Holmes and Doctor Watson,"[Mystery, Crime, Adventure]"
9179,149334,Nocturnal Animals,"[Drama, Thriller]"
9259,156605,Paterson,"[(no, genres, listed)]"
9367,162414,Moonlight,[Drama]
9448,167570,The OA,"[(no, genres, listed)]"
9514,171495,Cosmos,[Unknown]
9515,171631,Maria Bamford: Old Baby,"[Comedy, Documentary]"


In [292]:
year_list = [1,2,3,4,5,6]
for title in temp_for_year['title'].values: # 연도 결측치 영화에 대해 title 을 순회
    movies_df.loc[movies_df['title']==title, 'title'] = f"{title} (0)"
    
movies_df.loc[temp_for_year.index] # temp_for_year 를 이용해 변경 체크

,movieId,title,genres
6059,40697,Babylon 5 (0),[Sci-Fi]
9031,140956,Ready Player One (0),"[Action, Sci-Fi, Thriller]"
9091,143410,Hyena Road (0),"[War, Drama, Thriller, Action]"
9138,147250,The Adventures of Sherlock Holmes and Doctor Watson (0),"[Mystery, Crime, Adventure]"
9179,149334,Nocturnal Animals (0),"[Drama, Thriller]"
9259,156605,Paterson (0),"[(no, genres, listed)]"
9367,162414,Moonlight (0),[Drama]
9448,167570,The OA (0),"[(no, genres, listed)]"
9514,171495,Cosmos (0),[Unknown]
9515,171631,Maria Bamford: Old Baby (0),"[Comedy, Documentary]"


In [293]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9739,193585,Flint (2017),[Drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [294]:
movies_df = movies_df.reset_index(drop=True)
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9732,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9733,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9734,193585,Flint (2017),[Drama]
9735,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [295]:
ratings_df

,userId,movieId,rating,timestamp
0,1,1.0,4.0,964982703
1,1,3.0,4.0,964981247
2,1,6.0,4.0,964982224
3,1,47.0,5.0,964983815
4,1,50.0,5.0,964982931
...,...,...,...,...
100831,610,166534.0,4.0,1493848402
100832,610,168248.0,5.0,1493850091
100833,610,168250.0,5.0,1494273047
100834,610,168252.0,5.0,1493846352


In [296]:
movies_df['title'] = movies_df['title'].str.strip()
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9732,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9733,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9734,193585,Flint (2017),[Drama]
9735,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [297]:
movies_df[['title_only', 'year']] = movies_df['title'].str.rsplit(r' ', n=1, expand=True) # expand means DataFrame or Series
movies_df

,movieId,title,genres,title_only,year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",Toy Story,(1995)
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",Jumanji,(1995)
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",Grumpier Old Men,(1995)
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",Waiting to Exhale,(1995)
4,5,Father of the Bride Part II (1995),[Comedy],Father of the Bride Part II,(1995)
...,...,...,...,...,...
9732,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",Black Butler: Book of the Atlantic,(2017)
9733,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",No Game No Life: Zero,(2017)
9734,193585,Flint (2017),[Drama],Flint,(2017)
9735,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",Bungo Stray Dogs: Dead Apple,(2018)


In [298]:
title_duplicated_df = movies_df[movies_df['title_only'].duplicated()] # only title
title_duplicated_df

,movieId,title,genres,title_only,year
697,915,Sabrina (1954),"[Comedy, Romance]",Sabrina,(1954)
1032,1344,Cape Fear (1962),"[Crime, Drama, Thriller]",Cape Fear,(1962)
1369,1873,"Misérables, Les (1998)","[Crime, Drama, Romance, War]","Misérables, Les",(1998)
1419,1941,Hamlet (1948),[Drama],Hamlet,(1948)
1527,2059,"Parent Trap, The (1998)","[Children, Comedy, Romance]","Parent Trap, The",(1998)
...,...,...,...,...,...
9601,176413,Bliss (2012),[Drama],Bliss,(2012)
9615,177763,Murder on the Orient Express (2017),"[Crime, Drama, Mystery]",Murder on the Orient Express,(2017)
9686,184349,Elsa & Fred (2005),"[Comedy, Drama, Romance]",Elsa & Fred,(2005)
9691,184931,Death Wish (2018),"[Action, Crime, Drama, Thriller]",Death Wish,(2018)


In [299]:
movies_df[movies_df['movieId']==147250]

,movieId,title,genres,title_only,year
9134,147250,The Adventures of Sherlock Holmes and Doctor Watson (0),"[Mystery, Crime, Adventure]",The Adventures of Sherlock Holmes and Doctor Watson,(0)
